# See how fast we can compute the largest eigenvalue of the up Persistent Laplacian

The possible approaches are:
- Compute the entire matrix (in one of the dense ways, see that notebook) and then use a method to compute its largest eigenvalue.
    - Just compute all eigenvalues via standard methods (eigvalsh) (cubic, exact).
    - Use an iterative algorithm, Lanczos (quadratic).
- Use an iterative algorithm that takes in an operator $ x \mapsto \Delta^{K, L}_{up}x$. 
    - Can provide speedup if the operator computes the assignment faster than just doing the computation with the matrix.
- Exploit the form of the matrix as $\Delta^{K, L}_{up} = DZZ^TZ^T$ for example, to solve an easier problem instead. 

Arbitrary goal: how big of a matrix can we compute in under a minute?

In [1]:
import numpy as np
import gudhi
import tadasets
import scipy.linalg as la

%load_ext autoreload
%autoreload 2
from persistent_laplacian_methods.dense_matrix_methods import *
from persistent_laplacian_methods.dense_boundary_matrix_helpers import *

In [2]:
points = tadasets.torus(
    n=500,
    c=2.0,       
    a=1.0,       
    seed=42,
)

rips = gudhi.RipsComplex(
    points=points,
    max_edge_length=0.9,
)
simplex_tree = rips.create_simplex_tree(
    max_dimension=5,
)

In [3]:
D, levels, cuts = boundary_matrices(simplex_tree)

In [4]:
i, j = 2000, 3000
q = 3

print("Filtration length: ", len(cuts))
print_boundary_dimensions(D, cuts, i, j)

Filtration length:  4766
matrix             K_2000           K_3000
D_1           (500, 2000)      (500, 3000)
D_2          (2000, 3421)     (3000, 7756)
D_3          (3421, 3576)    (7756, 12604)
D_4          (3576, 2489)   (12604, 14107)
D_5          (2489, 1185)   (14107, 11182)


In [5]:
prepared = prepare_up_persistent_laplacian(
    D, cuts, q=q, i=i, j=j
)

L_q_i_j_naive = compute_schur_D(*prepared)

In [6]:
L_q_i_j_naive.shape

(3576, 3576)

In [7]:
## Baseline computation

def compute_largest_eigenvalue_eigh(up_persistent_laplacian):
    eigs = la.eigh(up_persistent_laplacian, eigvals_only=True)
    return eigs[-1]

compute_largest_eigenvalue_eigh(L_q_i_j_naive)

np.float64(17.547950454075057)

## Results

We can extrapolate cubically using $2.4(n / 3500)^3$ to estimate that for a $10,000  \times 10,000$ matrix the eigenvalue computation will take about a minute.
- For the Schur computation at this size, we need a pseudoinverse of a $9000 \times 9000$ matrix. It is also cubic, so it makes sense that it takes more than a minute.

Even using dense computation of all eigenvalues, the bottleneck is clearly the computation of the matrix itself at these sizes.
So to compute the largest eigenvalue for as large a matrix as possible in under one minute, we need to
- either use operator methods to avoid making the matrix,
- or get clever with the factorization and solve an easier problem.

## Approaches

Clever methods:
-  Compute just $BZ$ and its singular values. This still requires $Z$ calculation, which should be the bottleneck for the SVD approach, so expected to be slow.

Operator methods:
- Turn BZ into an operator directly (still requires Z calculation, so still expected to be slow).
- Make the operator BB^Tx + EE^Tx - ED^T LSQ(D, E^tx)